## Example 1: User management with JupyterHub

In [ ]:
from jupyterhub.user import User
from jupyterhub.orm import User as ORMUser
from datetime import datetime

# Example 1: User management with JupyterHub
print("Example 1: User management with JupyterHub")

# Create users for a data science team
users_data = [
    {"name": "alice", "admin": True},
    {"name": "bob", "admin": False},
    {"name": "charlie", "admin": False}
]

# Create ORM users and demonstrate user properties
orm_users = []
for user_data in users_data:
    # Create ORM user object
    orm_user = ORMUser(name=user_data["name"])
    orm_user.admin = user_data["admin"]
    orm_user.created = datetime.now()
    orm_user.last_activity = datetime.now()
    orm_users.append(orm_user)

# Display user information
print(f"Created {len(orm_users)} ORM users:")
for orm_user in orm_users:
    print(f"  {orm_user.name}: admin={orm_user.admin}")

# Demonstrate user properties
sample_user = orm_users[0]
print(f"\nUser properties for {sample_user.name}:")
print(f"  Created: {sample_user.created}")
print(f"  Admin status: {sample_user.admin}")
print(f"  Last activity: {sample_user.last_activity}")

print("Example 1 completed successfully")

## Example 2: Custom spawner configuration

In [ ]:
from jupyterhub.spawner import LocalProcessSpawner
from traitlets import Dict, Unicode

# Example 2: Custom spawner configuration
print("Example 2: Custom spawner configuration")

# Create a custom spawner class
class DataScienceSpawner(LocalProcessSpawner):
    """Custom spawner for data science environment"""
    
    # Custom environment variables
    environment = Dict(config=True, help="Environment variables for user servers")
    
    # Custom working directory  
    notebook_dir = Unicode("~/notebooks", config=True, help="Notebook directory")

# Configure the spawner with required attributes
spawner = DataScienceSpawner()
spawner.environment = {
    'CUDA_VISIBLE_DEVICES': '0',
    'OMP_NUM_THREADS': '4'
}
spawner.notebook_dir = "~/data-science-projects"

# Set required attributes to avoid AttributeError
spawner.hub = type('MockHub', (), {'public_host': 'localhost', 'ip': '127.0.0.1'})()
spawner.user = type('MockUser', (), {'name': 'testuser'})()

print(f"Custom spawner configured:")
print(f"  Class: {spawner.__class__.__name__}")
print(f"  Notebook dir: {spawner.notebook_dir}")
print(f"  Custom env vars: {len(spawner.environment)}")
print(f"  Hub host: {spawner.hub.public_host}")

print("Example 2 completed successfully")

## Example 3: Custom authentication system

In [ ]:
from jupyterhub.auth import Authenticator
from traitlets import Set

# Example 3: Custom authentication system
print("Example 3: Custom authentication system")

# Create a custom authenticator
class TeamAuthenticator(Authenticator):
    """Custom authenticator for team-based access"""
    
    # Define allowed teams
    allowed_teams = Set(config=True, help="Set of allowed team names")
    
    async def authenticate(self, handler, data):
        """Authenticate user based on team membership"""
        username = data.get('username', '')
        password = data.get('password', '')
        team = data.get('team', '')
        
        # Simple validation for demonstration
        if not username or not password:
            return None
            
        # Check team membership
        if self.allowed_teams and team not in self.allowed_teams:
            self.log.warning(f"User {username} not in allowed teams")
            return None
        
        # Simulate authentication success
        user_info = {
            'name': username,
            'auth_state': {
                'team': team,
                'access_level': 'standard'
            }
        }
        
        # Grant admin access to team leads
        if username.endswith('-lead'):
            user_info['admin'] = True
            user_info['auth_state']['access_level'] = 'admin'
        
        return user_info

# Configure the authenticator
auth = TeamAuthenticator()
auth.allowed_teams = {'data-team', 'ml-team', 'research-team'}

print(f"Custom authenticator configured:")
print(f"  Class: {auth.__class__.__name__}")
print(f"  Allowed teams: {list(auth.allowed_teams)}")

# Simulate authentication
test_users = [
    {'username': 'alice-lead', 'password': 'secret', 'team': 'data-team'},
    {'username': 'bob', 'password': 'secret', 'team': 'ml-team'}
]

print(f"\nTesting authentication:")
for user_data in test_users:
    print(f"  User: {user_data['username']} (team: {user_data['team']})")

print("Example 3 completed successfully")

## Example 4: Service integration and handlers

In [ ]:
from jupyterhub.handlers import BaseHandler
from tornado.web import authenticated
import json

# Example 4: Service integration and handlers
print("Example 4: Service integration and handlers")

# Create a custom API handler
class MetricsHandler(BaseHandler):
    """Custom handler for serving metrics data"""
    
    @authenticated
    async def get(self):
        """Get system metrics"""
        # Simulate metrics collection
        metrics = {
            'active_users': len(self.users) if hasattr(self, 'users') else 3,
            'running_servers': 5,
            'total_memory_gb': 128,
            'cpu_cores': 32,
            'uptime_hours': 24.5
        }
        
        self.set_header('Content-Type', 'application/json')
        self.write(json.dumps(metrics))
    
    @authenticated  
    async def post(self):
        """Update metrics configuration"""
        try:
            data = json.loads(self.request.body)
            # Process metrics update
            response = {
                'status': 'success',
                'updated_fields': list(data.keys())
            }
            self.write(json.dumps(response))
        except json.JSONDecodeError:
            self.set_status(400)
            self.write({'error': 'Invalid JSON payload'})

# Service configuration
service_config = {
    'name': 'metrics-service',
    'url': 'http://127.0.0.1:9000',
    'command': ['python', '-m', 'metrics_service'],
    'environment': {
        'METRICS_PORT': '9000',
        'LOG_LEVEL': 'INFO'
    }
}

print(f"Custom service configured:")
print(f"  Name: {service_config['name']}")
print(f"  URL: {service_config['url']}")
print(f"  Environment vars: {len(service_config['environment'])}")

# Handler registration pattern
handlers = [
    (r"/services/metrics/api", MetricsHandler),
]

print(f"\nAPI handlers registered:")
for pattern, handler_class in handlers:
    print(f"  {pattern} -> {handler_class.__name__}")

print("Example 4 completed successfully")

## Example 5: Hub application configuration

In [ ]:
from jupyterhub.app import JupyterHub
from traitlets.config import Config

# Example 5: Hub application configuration
print("Example 5: Hub application configuration")

# Create JupyterHub configuration
c = Config()

# Configure basic hub settings
c.JupyterHub.bind_url = 'http://0.0.0.0:8000'
c.JupyterHub.hub_ip = '127.0.0.1'
c.JupyterHub.hub_port = 8081

# Configure authentication
c.JupyterHub.authenticator_class = 'jupyterhub.auth.PAMAuthenticator'
c.Authenticator.admin_users = {'admin', 'root'}

# Configure spawner  
c.JupyterHub.spawner_class = 'jupyterhub.spawner.LocalProcessSpawner'
c.Spawner.start_timeout = 60
c.Spawner.notebook_dir = '~/notebooks'

# Create JupyterHub instance with configuration
hub = JupyterHub(config=c)

print(f"JupyterHub configured:")
print(f"  Bind URL: {hub.bind_url}")
print(f"  Hub IP: {hub.hub_ip}")
print(f"  Hub Port: {hub.hub_port}")
print(f"  Authenticator: {hub.authenticator_class}")
print(f"  Spawner: {hub.spawner_class}")

# Display configuration summary
print(f"\nConfiguration applied:")
print(f"  Hub configuration complete: True")
print(f"  Ready for deployment: True")

print("Example 5 completed successfully")